# Setting up the Blocks World Environment
We are aiming to train a neural network to act as a single-armed robot that can solve a Blocks World problem. For this setting we will have a set of 5 blocks $B = \{A, B, C, D, E\}$ with the goal state being the state where these blocks are stacked in a descending order.


## Predicates -
1. Each block can either be on top of any of the other block (`on(x,y)`) or on the table (`onTable(x)`). Thus 5 predicates per block. Thus in total $5 \times 4 + 5 = \boxed{25}$ predicates to denote the block's position.
2. Each of the blocks can either have some block on top of it or not. This can be denoted using a predicate like `clear(x)` $\forall x \in B$. Thus resulting in $\boxed{5}$ predicates.
3. The robot arm can either be empty or not, we can denote both states using a single predicate `armempty`. Thus $\boxed{1}$ more predicate.
4. If the robot arm is not empty, it must be holding a block. This requires new predicates `holding(x)` $\forall x \in B$. Thus resulting in $\boxed{5}$ predicates.

$$
\text{Total No. of Predicates} = 20 + 5 + 5 + 5 + 1 = 36
$$

## Actions -
1. `stack(x,y)` $\text{s.t} \ x \in B, y \in B-\{x\}$ can be used to stack a block `x` on top of some block `y`. This can have $\boxed{20}$ different permutations.
2. Similarly, `unstack(x,y)` $\text{s.t} \ x \in B, y \in B-\{x\}$ can be used to unstack a block `x` on top of some block `y` and will have also have $\boxed{20}$ permutations.
3. `pickup(x)` $\text{s.t} \ x \in B$ can be used to pick up any block from the table. This will have $\boxed{5}$ possible permutations.
4. Similarly, `putdown(x)` $\text{s.t} \ x \in B$ can be used to put any block on the table and will also have $\boxed{5}$ permutations.

$$
\text{Total No. of Actions} = 20 + 20 + 5 + 5 = 50
$$

# Actions Pre-conditions and Post-effect

1. `unstack(x,y)` -
    - Preconditions - `armempty`, `clear(x)`, `on(x,y)`
    - Add-Effects - `holding(x)`, `clear(y)`
    - Delete-Effects -`armempty`, `on(x,y)`, `clear(x)`

2. `stack(x,y)` -
    - Preconditions - `holding(x)`, `clear(y)`
    - Add-Effects -`on(x,y)`, `armempty`, `clear(x)`
    - Delete-Effects -`holding(x)`, `clear(y)`

3. `pickup(x)` -
    - Preconditions - `armempty`, `clear(x)`, `onTable(x)`
    - Add-Effects - `holding(x)`
    - Delete-Effects - `armempty`, `onTable(x)`, `clear(x)`

4. `putdown(x)` -
    - Preconditions - `holding(x)`
    - Add-Effects - `onTable(x)`, `armempty`, `clear(x)`
    - Delete-Effects - `holding(x)`

In [ ]:
from itertools import combinations
import random

import torch

In [ ]:
class StateGenerator:
  def __init__(self, blocks):
    self.blocks = blocks

  def convert_to_raw_state(self, blocks):
    return [blocks], None

  def partition(self, k=3, hold=False):
    blocks = random.sample(self.blocks, k=len(self.blocks))
    held_block = None

    if hold:
      held_block = blocks.pop()
      k -= 1

    n = len(blocks)

    if not (1 <= k <= n):
      lower_limit = 2 if hold else 1
      upper_limit = len(self.blocks)
      input_k     = k+1 if hold else k

      raise ValueError(f"k must be between {lower_limit} and {upper_limit}, got {input_k}")

    cuts = sorted(random.sample(range(1, n), k - 1))
    cuts = [0] + cuts + [n]
    piles = [blocks[cuts[i]:cuts[i + 1]] for i in range(k)]

    return piles, held_block


In [ ]:
class BlocksWorldEnv:
  def __init__(self, block_names: list[str], goal: list[str]):
    self.blocks = block_names
    self.state_generator = StateGenerator(self.blocks)

    # Making the predicates
    self.predicates = self._init_predicates()
    self.predicates_lookup = {pred: idx for idx, pred in enumerate(self.predicates)}

    # Making the actions
    self.actions = []
    self.action_precond: list[set[int]] = []
    self.action_add    : list[set[int]] = []
    self.action_del    : list[set[int]] = []
    self._init_actions()

    self.actions_lookup = {action: idx for idx, action in enumerate(self.actions)}

    # Goal state of the configuration
    raw_goal_state  = self.state_generator.convert_to_raw_state(goal)
    self.goal_state = self.convert_raw_to_predicates(raw_goal_state)
    self.goal_state = self.convert_predicates_to_state(self.goal_state)

    # Set current state of the environment
    self.initial_state = None
    self.state         = None
    self.exploration_actions       = []
    self.reset() # sets the initial state and the current state

  # ============================== For Predicates ==============================
  def _init_predicates(self):
    predicates = []

    # All on(x,y) predicates
    for x in self.blocks:
      for y in self.blocks:
        if x == y:
          continue

        predicate = f"on({x},{y})"
        predicates.append(predicate)

    # All onTable(x) predicates
    for x in self.blocks:
      predicate = f"onTable({x})"
      predicates.append(predicate)

    # All clear(x) predicates
    for x in self.blocks:
      predicate = f"clear({x})"
      predicates.append(predicate)

    # armempty predicate
    predicates.append("armempty")

    # All holding(x) predicates
    for x in self.blocks:
      predicate = f"holding({x})"
      predicates.append(predicate)

    return predicates

  # ============================== For Actions ==============================
  def _register_action(self, op_type, x, y = None):
    if op_type == "stack":
        name = f"stack({x},{y})"
        preconds = [f"holding({x})", f"clear({y})"]
        adds = [f"on({x},{y})", "armempty", f"clear({x})"]
        dels = [f"holding({x})", f"clear({y})"]

    elif op_type == "unstack":
        name = f"unstack({x},{y})"
        preconds = ["armempty", f"clear({x})", f"on({x},{y})"]
        adds = [f"holding({x})", f"clear({y})"]
        dels = ["armempty", f"on({x},{y})", f"clear({x})"]

    elif op_type == "pickup":
        name = f"pickup({x})"
        preconds = ["armempty", f"clear({x})", f"onTable({x})"]
        adds = [f"holding({x})"]
        dels = ["armempty", f"onTable({x})", f"clear({x})"]

    elif op_type == "putdown":
        name = f"putdown({x})"
        preconds = [f"holding({x})"]
        adds = [f"onTable({x})", "armempty", f"clear({x})"]
        dels = [f"holding({x})"]

    else:
        raise ValueError(f"Unknown action type: {op_type}")

    # Append the action name and map predicate strings to index sets
    self.actions.append(name)
    self.action_precond.append({self.predicates_lookup[p] for p in preconds})
    self.action_add.append({self.predicates_lookup[p] for p in adds})
    self.action_del.append({self.predicates_lookup[p] for p in dels})

  def _init_actions(self):
    # All stack(x) actions
    for x in self.blocks:
      for y in self.blocks:
        if x == y:
          continue

        self._register_action("stack", x, y)

    # All unstack(x) actions
    for x in self.blocks:
      for y in self.blocks:
        if x == y:
          continue

        self._register_action("unstack", x, y)

    # All pickup(x) actions
    for x in self.blocks:
      self._register_action("pickup", x)

    # All putdown(x) actions
    for x in self.blocks:
      self._register_action("putdown", x)

  # ============================== For Initial State Generation ==============================
  def convert_raw_to_predicates(self, raw_state):
    block_piles, held_block = raw_state
    state = set()

    if held_block:
      state.add(f"holding({held_block})")
    else:
      state.add("armempty")

    for pile in block_piles:
      bottom_block = pile[0]
      top_block    = pile[-1]

      state.add(f"onTable({bottom_block})")
      state.add(f"clear({top_block})")

      if len(pile) > 1:
        for idx in range(1,len(pile)):
          x = pile[idx]
          y = pile[idx-1]
          state.add(f"on({x},{y})")

    return state

  def convert_predicates_to_state(self, state):
    state = {self.predicates_lookup[p] for p in state}
    return state

  def convert_state_to_predicates(self, state):
    state = {self.predicates[id] for id in state}
    return state

  def convert_state_to_vector(self, state=None):
    if state is None:
      state = self.state

    output = torch.zeros(len(self.predicates))
    output[list(state)] = 1
    return output

  def generate_initial_state(self, hold=False):
    k_values = [1, 2, 3, 4, 5] if not hold else [2, 3, 4, 5]
    weights = [0.1, 0.8 / 3, 0.8 / 3, 0.8 / 3, 0.1] if not hold else [0.3, 0.3, 0.3, 0.1]
    k = random.choices(k_values, weights)[0]

    raw_state = self.state_generator.partition(k=k, hold=hold)
    state     = self.convert_raw_to_predicates(raw_state)
    state     = self.convert_predicates_to_state(state)
    return state

  # ============================== For State Transition ==============================
  def get_eligible_actions(self, state=None):
    if state is None:
      state = self.state

    eligible_actions = []

    for action_id, action_precondition in enumerate(self.action_precond):
      if action_precondition.issubset(state):
        eligible_actions.append(action_id)

    return eligible_actions

  def is_action_eligible(self, state, action):
    action_precondition = self.action_precond[action]
    return action_precondition.issubset(state)

  def convert_action_to_vector(self, actions):
    output = torch.zeros(len(self.actions))
    output[list(actions)] = 1
    return output

  def take_step(self, action):
    if not self.is_action_eligible(self.state, action):
      raise ValueError(f"Action {action} is not eligible for the state {self.state}")

    add_effect: set[int] = self.action_add[action]
    del_effect: set[int] = self.action_del[action]

    self.state = self.state.difference(del_effect).union(add_effect)
    self.exploration_actions.append(action)
    return self.state

  def reset(self):
    self.exploration_actions = []

    while True:
      hold_block = random.random() > 0.7
      self.initial_state = self.generate_initial_state(hold_block)
      self.state = self.initial_state

      if not self.is_goal_reached():
        break

    return self.state

  def is_goal_reached(self):
    return self.state == self.goal_state



In [ ]:
class RewardModel(BlocksWorldEnv):
  def __init__(self, block_names: list[str], goal: list[str]):
    super().__init__(block_names, goal)
    self.prev_match = None

    self.reset()

  def _calc_match(self):
    return len(self.state.intersection(self.goal_state))

  # Takes a step and calculates reward
  def take_step(self, action):
    super().take_step(action)

    curr_match = self._calc_match()
    reward     = curr_match - self.prev_match
    self.prev_match = curr_match

    goal_reached = self.is_goal_reached()

    return self.state, reward, goal_reached

  def reset(self):
    super().reset()
    self.prev_match = self._calc_match()
    return self.state


In [ ]:
blocks = ["A", "B", "C", "D", "E"]
goal   = ["E", "D", "C", "B", "A"]

env = RewardModel(blocks, goal)

for i in range(6):
  print("========== State =========")
  for id in env.state:
    print(env.predicates[id])

  print("========== Actions =========")
  action = env.get_eligible_actions()[0]
  print(env.actions[action])
  env.take_step(action)

# env.convert_state_to_vector()
# env.convert_action_to_vector(actions)

========== State =========
on(A,E)
on(B,A)
on(C,D)
on(E,C)
onTable(D)
clear(B)
armempty
========== Actions =========
unstack(B,A)
========== State =========
holding(B)
on(E,C)
on(A,E)
onTable(D)
clear(A)
on(C,D)
========== Actions =========
stack(B,A)
========== State =========
on(E,C)
on(A,E)
clear(B)
on(B,A)
onTable(D)
on(C,D)
armempty
========== Actions =========
unstack(B,A)
========== State =========
holding(B)
on(E,C)
on(A,E)
onTable(D)
clear(A)
on(C,D)
========== Actions =========
stack(B,A)
========== State =========
on(E,C)
on(A,E)
clear(B)
on(B,A)
onTable(D)
on(C,D)
armempty
========== Actions =========
unstack(B,A)
========== State =========
holding(B)
on(E,C)
on(A,E)
onTable(D)
clear(A)
on(C,D)
========== Actions =========
stack(B,A)
